# การตรวจจับไฟป่าจากภาพถ่ายทางอากาศด้วยคอมพิวเตอร์วิทัศน์ (computer vision) และอากาศยานไร้คนขับ (drone)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/main.ipynb)

**เริ่มต้นที่นี่** โน้ตบุ๊กฉบับนี้คือคู่มือเดินเรื่องของทั้งโปรเจ็ค ตั้งแต่ว่ากำลังสร้างอะไร ระบบประกอบกันขึ้นมาอย่างไร
ชุดข้อมูลได้มาอย่างไร ไปจนถึงว่าโค้ดที่รันได้จริงแต่ละส่วนอยู่ตรงไหน

เอกสารฉบับนี้คือ *แผนที่* ส่วน [`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb) คือ *พื้นที่จริง*
ซึ่งรวมทั้งสามขั้นตอนไว้ในไฟล์เดียว รันจากบนลงล่างได้ตลอด

| ส่วน | เนื้อหา |
|---|---|
| [ส่วนที่ 1](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-1) | เทรนโมเดลตรวจจับไฟด้วยชุดข้อมูลจาก Roboflow |
| [ส่วนที่ 2](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2) | รันโมเดลที่เทรนแล้วกับภาพนิ่งหนึ่งภาพ |
| [ส่วนที่ 3](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3) | ตรวจจับและติดตามวัตถุด้วย ByteTrack ตลอดทั้งวิดีโอ |

> **หมายเหตุ** ฉบับภาษาไทยในโฟลเดอร์ `th/` รวมทั้งสามส่วนไว้ในโน้ตบุ๊กไฟล์เดียว
> ส่วนฉบับภาษาอังกฤษที่รากของคลังโค้ดยังแยกเป็นสามไฟล์อยู่ โค้ดที่รันเหมือนกันทุกประการ
> ต่างกันแค่การจัดวางไฟล์ คำอธิบาย และคอมเมนต์

อ่านเอกสารฉบับนี้ไล่จากต้นจนจบ ระหว่างทางจะบอกชัดเจนว่าถึงจุดไหนแล้วควรเปิดโน้ตบุ๊กส่วนไหนขึ้นมารันประกอบ

*เอกสารฉบับนี้เรียบเรียงตามโปรเจ็คในบทความของ Roboflow เรื่อง
[Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/)
โดย Timothy M. (รายการอ้างอิงฉบับเต็มอยู่ท้ายเอกสาร) ทั้งนี้ ผู้เขียนปรับโค้ดทั้งหมดให้ทันสมัยแล้ว
เพราะบทความต้นฉบับเขียนขึ้นบนไลบรารีรุ่นปี 2023 ซึ่งรันไม่ได้อีกต่อไป*

## ทำไมต้องตรวจจับไฟป่าจากทางอากาศ

การตรวจจับไฟป่าแต่เดิมพึ่งอยู่สองอย่าง คือเซนเซอร์ภาคพื้นดินกับภาพถ่ายดาวเทียม ซึ่งทั้งคู่มีข้อจำกัดจริง ๆ ของตัวเอง
เซนเซอร์ภาคพื้นดินเห็นเฉพาะจุดที่ไปติดตั้งไว้แล้วเท่านั้น และการติดให้ทั่วทั้งผืนป่าก็มีต้นทุนสูงมาก
ส่วนดาวเทียมครอบคลุมพื้นที่ได้กว้างกว่ามาก แต่ด้วยความละเอียดเชิงพื้นที่ (spatial resolution)
และรอบเวลาการถ่ายซ้ำ (revisit time) ที่มีอยู่ ก็ยังพลาดไฟในช่วงที่มันยังเล็กพอจะดับได้ด้วยต้นทุนต่ำ

ช่วงเวลานั้นคือทั้งหมดของเกมนี้ ไฟที่เจอในไม่กี่นาทีแรก กับไฟกองเดียวกันที่เจอเมื่อผ่านไปแล้วหนึ่งชั่วโมง
คือคนละปัญหากันเลย

อากาศยานไร้คนขับที่ติดกล้องและมีโมเดลคอมพิวเตอร์วิทัศน์อยู่ด้วย จึงเข้ามาอุดช่องว่างระหว่างสองแนวทางนั้น
มันกวาดพื้นที่ได้เร็วกว่าคนเดินเท้ามาก บินต่ำพอจะเห็นรายละเอียดที่ดาวเทียมมองไม่เห็น
และสั่งบินซ้ำตามตารางเหนือภูมิประเทศที่ไม่มีทางจัดคนไปลาดตระเวนได้ทุกวันก็ยังได้

## ระบบนี้ต้องใช้อะไรบ้าง

สี่อย่าง แบ่งเป็นฮาร์ดแวร์สองและซอฟต์แวร์สอง

| | องค์ประกอบ | หน้าที่ |
|---|---|---|
| 🛩️ | **อากาศยานไร้คนขับ** | พาเซนเซอร์ขึ้นบินเหนือภูมิประเทศที่ลาดตระเวนด้วยเท้าแล้วช้าหรือเสี่ยงเกินไป |
| 📷 | **โมดูลกล้อง WiFi** | บันทึกภาพและส่งสัญญาณกลับลงมายังภาคพื้นดิน |
| 🏷️ | **บัญชี Roboflow** | เก็บชุดข้อมูล ตัดเฟรมภาพจากวิดีโอ และมีเครื่องมือติดป้ายกำกับให้ |
| 📓 | **Google Colab** | GPU ฟรีสำหรับเทรนโมเดล โน้ตบุ๊กทุกไฟล์ในคลังโค้ดนี้รันบนนั้น |

## ระบบทำงานอย่างไรตั้งแต่ต้นจนจบ

การตรวจจับเป็นแค่ช่วงกลางของเรื่อง คุณค่าจริง ๆ อยู่ที่สิ่งที่เกิดขึ้นสองฝั่งของมัน
คือการพากล้องไปอยู่เหนือพื้นที่ที่ถูกต้อง และการทำให้คนเคลื่อนเข้าไปรับมือได้จริงหลังจากเจอไฟแล้ว

```
   ┌──────────────┐
   │  1. DRONE    │
   │   DEPLOYED   │
   └──────┬───────┘
          │  video / stills
          ▼
   ┌──────────────┐
   │ 2. REMOTE    │
   │  INSPECTION  │
   └──────┬───────┘
          │  frames
          ▼
   ┌──────────────┐
   │ 3. COMPUTER  │  ← ส่วนที่โปรเจ็คนี้สร้างขึ้นจริง
   │    VISION    │
   └──────┬───────┘
          │  detections + track IDs
          ▼
   ┌──────────────┐
   │ 4. FIRE      │
   │  DETECTED    │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 5. CONTROL   │
   │   CENTRE     │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 6. RESPONSE  │
   │  TEAM SENT   │
   └──────────────┘
```

| ขั้น | ขั้นตอน | รายละเอียด |
|---|---|---|
| 1 | Drone deployed | ปล่อยอากาศยานบินเหนือพื้นที่สำรวจ กวาดพื้นที่ได้เร็วและเข้าถึงภูมิประเทศที่ลาดตระเวนยาก |
| 2 | Remote inspection | เจ้าหน้าที่ที่ผ่านการฝึกบังคับและเฝ้าดูอากาศยานจากสถานีภาคพื้นดิน |
| 3 | Computer vision | โมเดล YOLO26 ไล่ดูทุกเฟรมเพื่อหาเปลวไฟและกลุ่มควัน |
| 4 | Fire detected | ผลตรวจจับที่ยืนยันแล้วจะยิงการแจ้งเตือนทันที |
| 5 | Control centre | ผู้ปฏิบัติงานประเมินการแจ้งเตือนแล้วตัดสินใจว่าจะรับมืออย่างไร |
| 6 | Response team sent | ส่งชุดปฏิบัติการเข้าไปยังพิกัดที่ตรวจพบ |

ขั้นที่ 1, 2, 5 และ 6 เป็นเรื่องของการปฏิบัติการ ทั้งอากาศยาน คน และระเบียบวิธี
**ขั้นที่ 3 คือส่วนที่คลังโค้ดนี้สร้างขึ้นจริง** ส่วนขั้นที่ 4 เป็นผลพวงโดยตรงจากขั้นที่ 3
โมเดลมองหาร่องรอยเชิงภาพของไฟ ได้แก่ เปลวไฟที่มองเห็นได้ กลุ่มควัน และสีที่เปลี่ยนไปของพื้นที่ที่กำลังไหม้

ขั้นตอนการสร้างที่พาไปถึงจุดนั้นมีสี่ขั้น

1. **เตรียมชุดข้อมูล** รวบรวมภาพถ่ายทางอากาศดิบ
2. **ติดป้ายกำกับแล้วสร้างเวอร์ชันของชุดข้อมูล** วาดกรอบ แล้วตรึงผลลัพธ์เอาไว้
3. **เทรนโมเดล** โน้ตบุ๊กไฟล์ที่ 1
4. **ทดสอบโมเดล** โน้ตบุ๊กไฟล์ที่ 2 และ 3

## ขั้นที่ 1 เตรียมชุดข้อมูล

ต้นทางคือ **FLAME dataset** (*Aerial Imagery Pile burn detection using drones (UAVs)*, IEEE Dataport)
ซึ่งเป็นภาพถ่ายทางอากาศของการเผากองเชื้อเพลิงแบบมีการควบคุม ถือว่าใกล้เคียงเป้าหมายจริงมากที่สุดเท่าที่ชุดข้อมูลสาธารณะจะให้ได้

ไฟล์ที่ได้มาเป็นวิดีโอ (`.mp4`) ไม่ใช่ภาพนิ่ง ซึ่งเป็นรูปแบบตามธรรมชาติของการถ่ายด้วยอากาศยานไร้คนขับ
แต่เป็นรูปแบบที่ผิดสำหรับการเทรนโมเดลตรวจจับวัตถุ (object detection) ที่ต้องการภาพรายเฟรมพร้อมป้ายกำกับ

Roboflow จัดการแปลงให้ เพียงอัปโหลดวิดีโอ เลือกอัตราการตัดเฟรม ระบบก็จะซอยวิดีโอออกมาเป็นภาพนิ่งทีละใบ

อัตราการตัดเฟรมสำคัญกว่าที่เห็น ถ้าตั้งสูงเกินไปจะได้ภาพที่เกือบเหมือนกันเป็นพันใบ
ซึ่งทำให้ชุดข้อมูลบวมขึ้นโดยไม่ได้สารสนเทศอะไรเพิ่ม และที่แย่กว่านั้นคือภาพที่เกือบซ้ำกันอาจกระจายคร่อมเส้นแบ่ง
ระหว่างชุดเทรน (train) กับชุดตรวจสอบ (validation) แล้วดันคะแนนฝั่ง validation ให้สูงเกินจริงแบบเงียบ ๆ
ถ้าตั้งต่ำเกินไปก็จะเสียความหลากหลายเชิงภาพ ซึ่งเป็นตัวที่ทำให้โมเดลวางนัยทั่วไป (generalise) ได้

## ขั้นที่ 2 ติดป้ายกำกับและสร้างเวอร์ชันของชุดข้อมูล

ทุกเฟรมถูกติดป้ายด้วยกรอบล้อมวัตถุ (bounding box) สำหรับงานตรวจจับวัตถุ ผ่านเครื่องมือติดป้ายกำกับของ Roboflow
โดยโปรเจ็คนี้ใช้ **ชั้นข้อมูลเดียว คือ `fire`** วาดครอบทุกจุดที่เห็นไฟในเฟรม

การใช้โมเดลชั้นข้อมูลเดียวเป็นการลดความซับซ้อนแบบตั้งใจ มันเลี่ยงคำถามที่ยากที่สุดของการติดป้ายกำกับไปเลย
นั่นคือตกลงแล้วเปลวไฟจบตรงไหนและควันเริ่มตรงไหน โดยแลกกับการที่โมเดลแยกสองอย่างนี้ออกจากกันไม่ได้ตอน inference
แต่สำหรับระบบเตือนภัยล่วงหน้าที่ต้องการผลลัพธ์แค่ว่า "ตรงนี้มีอะไรกำลังไหม้ ส่งคนไปดู" การแลกแบบนี้ถือว่าคุ้ม

พอติดป้ายเสร็จ ชุดข้อมูลจะถูกตรึงไว้เป็น **เวอร์ชัน (version)** ซึ่งเป็นสำเนาที่แก้ไม่ได้
มีการแบ่ง train/validation/test และการตั้งค่าประมวลผลข้อมูลเบื้องต้น (preprocessing) เป็นของตัวเอง
เวอร์ชันนี่แหละคือสิ่งที่ทำให้การเทรนทำซ้ำได้ (reproducible) กล่าวคือสั่ง `version(1)` ปีหน้าก็ยังได้ข้อมูลชุดเดิมกลับมา
โปรเจ็คนี้เทรนด้วยเวอร์ชันที่ 1 ของโปรเจ็ค `drone-fire-detection-byija`

## ขั้นที่ 3 เทรนโมเดล

พอมีเวอร์ชันของชุดข้อมูลที่ติดป้ายแล้วอยู่ในมือ การเทรนมีแค่สองจังหวะ คือดึงชุดข้อมูลลงมาจาก Roboflow
แล้วปรับละเอียด (fine-tune) ต่อจาก checkpoint ของ YOLO ที่ผ่านการเทรนล่วงหน้ามาแล้ว

ดึงชุดข้อมูล

```python
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")
dataset = project.version(1).download("yolov8")
```

แล้วเทรน

```python
from ultralytics import YOLO

model = YOLO("yolo26m.pt")
train_results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=800,
    plots=True,
)
```

โค้ดข้างบนมีสองจุดที่ต้องอธิบายเพิ่ม เพราะต่างจากบทความต้นฉบับทั้งคู่

- **`download("yolov8")` เป็นชื่อรูปแบบการจัดวางไฟล์ของชุดข้อมูล ไม่ใช่ชื่อโมเดล**
  มันหมายถึง "ภาพ + ไฟล์ label `.txt` ตามมาตรฐาน YOLO + `data.yaml`" ซึ่งเป็นรูปแบบที่ YOLO26 อ่านได้เลยโดยไม่ต้องแก้อะไร
  ตัวเลขเวอร์ชันในสตริงนั้นไม่เกี่ยวอะไรกับเวอร์ชันของโมเดลที่เอามาเทรน
- **ต้นฉบับใช้ `yolov8m.pt` ผ่าน CLI `yolo task=detect mode=train`** ส่วนคลังโค้ดนี้เทรน **YOLO26**
  ซึ่งเป็นสถาปัตยกรรมแบบครบวงจร (end-to-end) และไม่ผ่าน NMS โดยเรียกผ่าน Python API
  ผลในทางปฏิบัติอย่างหนึ่งคือ ไม่เหลือค่าขีดแบ่ง `iou` ของ NMS ให้ต้องปรับอีกต่อไป

`imgsz=800` ถือว่าใหญ่ และตั้งใจให้ใหญ่ เพราะไฟในภาพถ่ายทางอากาศมักกินพื้นที่เฟรมแค่นิดเดียว
การย่อภาพลงเหลือ 640 ก็คือการทิ้งพิกเซลส่วนที่สำคัญที่สุดไปพอดี ข้อแลกเปลี่ยนคือกินหน่วยความจำมากขึ้น
ถ้า GPU เริ่มบ่น ให้ลดเหลือ 640 หรือเปลี่ยนไปใช้ `yolo26n.pt`

### ผลลัพธ์ที่ดีหน้าตาเป็นอย่างไร

การเทรนด้วย YOLOv8 ในบทความต้นฉบับรายงานผลบนชั้นข้อมูล `fire` ไว้ดังนี้

| ตัวชี้วัด | ค่าที่ได้ |
|---|---|
| Recall | 0.95 |
| mAP@50 | 0.99 |
| mAP@50-95 | 0.63 |

ควรอ่านค่าทั้งสามรวมกัน ไม่ใช่แยกทีละตัว ค่า Recall 0.95 และ mAP@50 0.99 บอกว่าโมเดล *หาไฟเจอ* อย่างสม่ำเสมอ
ซึ่งเป็นเรื่องที่สำคัญที่สุดของระบบเตือนภัยล่วงหน้า เพราะการพลาดไฟที่เกิดขึ้นจริงไปหนึ่งครั้ง (false negative)
แพงกว่าการแจ้งเตือนผิด (false positive) มาก

ส่วนค่าที่ตกลงมาเหลือ 0.63 ที่เกณฑ์ mAP@50-95 บอกว่ากรอบวางไว้ *ประมาณ* ถูก แต่ยังไม่ *เป๊ะ*
ซึ่งคาดไว้อยู่แล้วและไม่ใช่เรื่องร้ายแรงในบริบทนี้ เพราะไฟไม่มีขอบที่ชัดเจนให้ตกลงกันได้ตั้งแต่แรก
คะแนนที่วัดด้วย IoU สูง ๆ จึงวัดความกำกวมของป้ายกำกับ (label ambiguity) พอ ๆ กับที่วัดข้อบกพร่องของตัวโมเดลเอง

ให้ถือค่าชุดนี้เป็นเกณฑ์ไว้เทียบ ไม่ใช่ค่าที่การันตีได้ เพราะตัวเลขที่ได้ย่อมขึ้นอยู่กับเวอร์ชันของชุดข้อมูลและวิธีแบ่งข้อมูลที่ใช้

---

### ▶ รัน `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 1 จาก 3 (เทรนโมเดล)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)

**เทรนโมเดลตรวจจับ**

ไฟล์นี้ดาวน์โหลดชุดข้อมูลจาก Roboflow ปรับละเอียด YOLO26 เป็นเวลา 50 epochs ตรวจสอบความแม่นยำของผลลัพธ์
ลองทำนายบนชุดทดสอบที่กันไว้ แล้วส่งออกไฟล์ weights

ปิดท้ายด้วยการดาวน์โหลด **`best.pt`** ซึ่งคือ checkpoint ที่เทรนเสร็จแล้ว
โน้ตบุ๊กอีกสองไฟล์ที่เหลือต้องใช้ไฟล์นี้ และไฟล์นี้ *ไม่ได้* เก็บไว้ในคลังโค้ด เพราะฉะนั้นเก็บรักษาไว้ให้ดี

> **ก่อนเริ่ม** สลับ Colab ไปใช้ runtime แบบ GPU (`Runtime` → `Change runtime type` → **T4 GPU**) แล้วเอา
> [Roboflow API key](https://app.roboflow.com/settings/api) แบบไม่มีค่าใช้จ่าย ไปใส่ไว้ในแผง 🔑 **Secrets**
> ของ Colab ภายใต้ชื่อ `ROBOFLOW_API_KEY`

---

## ขั้นที่ 4 นำไปใช้งานและทดสอบ

checkpoint ที่เทรนเสร็จแล้วไปต่อได้หลายทาง จะส่ง weights กลับขึ้น Roboflow แล้วให้บริการผ่าน hosted inference API
ของแพลตฟอร์มก็ได้ หรือจะรันบนฮาร์ดแวร์ของตัวเองด้วย [`roboflow/inference`](https://github.com/roboflow/inference) ก็ได้
ซึ่งเป็นทางเลือกที่สมเหตุสมผลกว่าสำหรับสถานีภาคพื้นดินของอากาศยานไร้คนขับ
เพราะการส่งทุกเฟรมไป-กลับกับ cloud API ไม่เร็วและไม่เสถียรพอ

แต่ถ้าเป้าหมายคือดูว่าโมเดลใช้ได้จริงหรือไม่ ทางที่ง่ายที่สุดคือโหลด `best.pt` เข้าโน้ตบุ๊กตรง ๆ แล้วดูผลด้วยตา
ซึ่งเป็นสิ่งที่โน้ตบุ๊กอีกสองไฟล์ถัดไปทำ โดยใช้ **[Supervision](https://supervision.roboflow.com/)**
แปลงผลลัพธ์ดิบจากโมเดลให้กลายเป็นภาพและวิดีโอที่วาดกรอบไว้แล้ว

## รัน inference บนภาพนิ่ง

เส้นทางฝั่งภาพนิ่งสั้นมาก อ่านภาพ รันโมเดล แปลงผลลัพธ์เป็นออบเจกต์ `Detections` ของ Supervision แล้ววาด

```python
import cv2
import supervision as sv
from ultralytics import YOLO

model = YOLO("best.pt")
image = cv2.imread("fire_image.png")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
```

`sv.Detections.from_ultralytics` คือรอยต่อที่มีประโยชน์ที่สุดในโค้ดชุดนี้ มันจัดรูปผลลัพธ์ของ YOLO
ให้เป็นโครงสร้างข้อมูลที่ annotator กินได้ พร้อมพาชื่อชั้นข้อมูลติดมาด้วย
และเมื่อเริ่มใช้การติดตามวัตถุ ก็จะพาหมายเลข track ติดมาด้วยเช่นกัน

จากนั้นค่อยวาดผลลัพธ์ ตรงนี้แหละคือจุดที่โค้ดในบทความ *พังจริง* ไม่ใช่แค่เก่า

```python
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))
```

บทความส่ง `labels=` เข้าไปที่ `BoxAnnotator` แต่ **อาร์กิวเมนต์นั้นถูกถอดออกตั้งแต่ supervision 0.22**
ตอนนี้การวาดถูกแยกเป็น `BoxAnnotator` (วาดกรอบ) กับ `LabelAnnotator` (วาดข้อความกำกับ)
ส่วนชื่อชั้นข้อมูลก็อ่านมาจาก `detections["class_name"]` ซึ่งได้ค่ามาจากตัวโมเดลเอง
แทนที่จะไปเปิดเทียบกับรายชื่อที่เขียนมือไว้ ซึ่งมีสิทธิ์หลุดไม่ตรงกับ weights ที่ใช้อยู่

---

### ▶ รัน `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 2 จาก 3 (ภาพนิ่ง)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2)

**ตรวจจับไฟบนภาพนิ่งหนึ่งภาพ**

ส่วนนี้โหลด `best.pt` รันกับภาพตัวอย่าง `fire_image.png` ในคลังโค้ดนี้ วาดผลตรวจจับด้วย Supervision แล้วบันทึกผลลัพธ์

เป็นวิธีที่เร็วที่สุดในการเช็กว่า checkpoint ที่เพิ่งเทรนเสร็จใช้ได้จริงหรือไม่

> **ต้องมี `best.pt`** จากส่วนที่ 1 ถ้ารันต่อกันมาในเซสชันเดียวจะหยิบไปใช้ให้เอง
> ถ้าเปิดโน้ตบุ๊กขึ้นมาใหม่แล้วข้ามมาส่วนนี้เลย โน้ตบุ๊กจะแจ้งให้อัปโหลด
> ส่วนนี้ไม่จำเป็นต้องใช้ GPU เพราะ inference กับภาพเดียวรันบน CPU ก็ทันใจ

---

## รัน inference บนวิดีโอ

ฝั่งวิดีโอเริ่มน่าสนใจ เพราะแค่ตรวจจับอย่างเดียวไม่พอ ถ้ารันโมเดลตรวจจับไล่ไปทีละเฟรม
สิ่งที่ได้คือกรอบชุดหนึ่งต่อหนึ่งเฟรม โดยไม่มีอะไรบอกเลยว่าไฟในเฟรมที่ 200 เป็นไฟกองเดียวกับในเฟรมที่ 199
เมื่อไม่มีความคงอยู่ของวัตถุ (object permanence) ก็นับจำนวนจุดไฟที่แยกจากกันไม่ได้
วัดไม่ได้ว่าไฟจุดหนึ่งไหม้มานานแค่ไหน และตามไม่ได้ว่ามันกำลังลุกลามหรือเปล่า

**ByteTrack** เข้ามาเติมความต่อเนื่องตรงนั้น แนวคิดหลักของมันควรทำความเข้าใจไว้ เพราะมันกำหนดหน้าตาของโค้ดที่ตามมา
กล่าวคือ ตัวติดตามวัตถุส่วนใหญ่จะทิ้งผลตรวจจับที่ค่าความเชื่อมั่น (confidence score) ต่ำไปก่อนเข้าขั้นตอนจับคู่
แต่ ByteTrack เก็บมันไว้ แล้วลองเอาไปจับคู่กับ track ที่สร้างไว้แล้วจากเฟรมก่อนหน้า
ไฟที่ถูกควันบังชั่วขณะหรือถูกบังตอนอากาศยานเอียงตัว ค่าความเชื่อมั่นจะตกลง แต่ไม่ได้แปลว่ามันหายไปจากฉาก
และกรอบที่มั่นใจน้อยแต่ตำแหน่งดันตรงกับ track ที่มั่นใจสูงจากเฟรมก่อนหน้า ก็แทบจะเป็นของจริงแน่นอน

### ตรงนี้คือจุดที่โค้ดต้นฉบับรันไม่ได้อีกต่อไป

เวอร์ชันที่เผยแพร่ในบทความติดตั้ง ByteTrack แบบนี้

```python
!git clone https://github.com/ifzhang/ByteTrack.git
!sed -i 's/onnx==1.8.1/onnx==1.9.0/g' requirements.txt
!pip3 install -q -r requirements.txt
!python3 setup.py -q develop
!pip install -q cython_bbox onemetric loguru lap thop
```

คือ build YOLOX จากซอร์ส บวก `onemetric` กับ `cython_bbox` บวก `sed` อีกสองบรรทัดที่ไปเลี่ยงบั๊กซึ่งต้นทางปิดไปนานแล้ว
ชุดเครื่องมือแบบนี้ build บน Python รุ่นปัจจุบันไม่ผ่านแล้ว

และมันก็ไม่จำเป็นอีกต่อไปด้วย เพราะ **ByteTrack มากับ Ultralytics อยู่แล้ว**
บล็อกทั้งหมดข้างบนจึงยุบเหลืออาร์กิวเมนต์เดียว

```python
result = model.track(
    frame,
    conf=0.1,
    persist=True,
    tracker="bytetrack.yaml",
    verbose=False,
)[0]
detections = sv.Detections.from_ultralytics(result)
detections = detections[detections.confidence >= 0.25]
```

`persist=True` คือตัวที่อุ้มสถานะของตัวติดตามข้ามการเรียกแต่ละครั้ง หมายเลข id จึงคงที่ข้ามเฟรม
ส่วน `from_ultralytics` อ่านหมายเลขเหล่านั้นเข้ามาไว้ใน `detections.tracker_id`

สังเกตค่าความเชื่อมั่นสองค่าที่ไม่เท่ากันในโค้ดข้างบน นั่นคือแนวคิดของ ByteTrack ที่แปลงเป็นโค้ดตรง ๆ
**ติดตามที่ `0.1` แต่แสดงผลที่ `0.25`** การป้อนเฉพาะกรอบที่มั่นใจสูงให้ตัวติดตาม
ก็คือการทิ้งผลตรวจจับที่มั่นใจน้อยซึ่งเป็นวัตถุดิบที่มันใช้กู้ track กลับมาพอดี
ฉะนั้นให้ติดตามจากทุกอย่างก่อน แล้วค่อยกรองตอนวาด

อีกเรื่องที่ควรรู้ไว้ถ้าจะเอาไปเทียบกับบทความ คือลูปเรนเดอร์ของบทความสร้างออบเจกต์ `BYTETracker` ขึ้นมา
แล้วไม่เคยเรียกใช้มันในลูปเลย วิดีโอผลลัพธ์ที่ได้จึงเป็นแค่ผลตรวจจับรายเฟรมที่ไม่มีการติดตามใด ๆ

---

### ▶ รัน `drone_fire_detection_yolo26.ipynb` — ส่วนที่ 3 จาก 3 (วิดีโอ)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [ดูบน GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3)

**ตรวจจับและติดตามไฟตลอดทั้งวิดีโอ**

ส่วนนี้รัน YOLO26 คู่กับ ByteTrack บน `fire.mp4` ไล่ไปทีละเฟรม วาดกรอบ ป้ายกำกับ หมายเลข track
และเส้นร่องรอยการเคลื่อนที่ แล้วเขียนออกมาเป็นวิดีโอผลลัพธ์

นอกจากนี้ยัง re-encode ไฟล์ผลลัพธ์เป็น H.264 ด้วย ffmpeg เพื่อให้เล่นได้ในหน้าโน้ตบุ๊ก
เพราะ Supervision เขียนออกมาเป็น `mp4v` ซึ่งเบราว์เซอร์ไม่ยอมถอดรหัสให้
นี่คือสาเหตุที่วิดีโอผลลัพธ์ในบทความต้นฉบับดูเหมือน "เล่นไม่ได้"

> **ต้องมี `best.pt`** จากส่วนที่ 1 และต้องใช้ runtime แบบ **T4 GPU**
> เพราะ inference วิดีโอบน CPU ช้าจนใช้งานจริงไม่ไหว

---

## ส่วนเสริม ตรวจสภาพแวดล้อมก่อนเริ่ม

เนื้อหาข้างบนไม่มีอะไรต้องรัน เพราะเอกสารฉบับนี้เป็นแผนที่ ไม่ใช่ไปป์ไลน์
แต่ถ้าอยากเช็กว่า runtime ที่ใช้อยู่พร้อมแล้วก่อนกระโดดไปส่วนที่ 1 ก็รันเซลล์ด้านล่างได้

In [ ]:
import shutil
import subprocess
import sys

print(f"Python {sys.version.split()[0]}\n")

gpu = shutil.which("nvidia-smi")
if gpu:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"GPU: {out or 'ตรวจพบแล้ว'}")
else:
    print("GPU: ไม่พบ ให้ตั้งค่าที่ Runtime > Change runtime type > T4 GPU")

print("ffmpeg:", "พร้อมใช้งาน" if shutil.which("ffmpeg") else "ไม่พบ (โน้ตบุ๊กไฟล์ที่ 3 ต้องใช้)")

for pkg in ("ultralytics", "supervision", "roboflow"):
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'ติดตั้งแล้ว')}")
    except ImportError:
        print(f"{pkg}: ยังไม่ได้ติดตั้ง (โน้ตบุ๊กแต่ละไฟล์ติดตั้งของตัวเอง)")

## อะไรเปลี่ยนไปบ้างหลังบทความต้นฉบับเผยแพร่

บทความเขียนขึ้นเมื่อเดือนกันยายน 2023 โค้ดในนั้นอ้างอิงไลบรารีที่พัฒนาต่อไปไกลแล้ว
และบางจุดก็ไกลพอจนรันไม่ได้อีกต่อไป ทุกอย่างในคลังโค้ดนี้จึงได้รับการปรับใหม่ ดังนี้

| | บทความ (2023) | คลังโค้ดนี้ |
|---|---|---|
| โมเดล | `yolov8m.pt` | **`yolo26m.pt`** แบบครบวงจร ไม่ผ่าน NMS |
| Ultralytics | `==8.0.20` | `>=8.4.122` |
| Supervision | `==0.1.0` | `>=0.30.0` |
| Roboflow | ไม่ได้ตรึงเวอร์ชัน | `>=1.4.1` |
| การเทรน | CLI `yolo task=detect mode=train` | Python API ร่วมกับ `results.save_dir` |
| การวาดผลลัพธ์ | `BoxAnnotator(labels=...)` | `BoxAnnotator` + `LabelAnnotator` |
| การติดตามวัตถุ | โคลน ByteTrack แล้ว build YOLOX จากซอร์ส | มากับ Ultralytics อยู่แล้ว |
| API key | `api_key="YOUR_API_KEY"` ในเซลล์ | Colab Secrets |
| วิดีโอผลลัพธ์ | `mp4v` (เบราว์เซอร์ถอดรหัสไม่ได้) | re-encode เป็น H.264 |

นอกจากนี้โน้ตบุ๊กชุดเดิมยังเปิดไม่ขึ้นด้วย ซึ่งมาจากสองสาเหตุที่เป็นคนละเรื่องกัน คือ
โน้ตบุ๊กสำหรับวิดีโอมีบล็อก `metadata.widgets` ที่ขาดคีย์ `state` ซึ่งเป็นเงื่อนไขที่ทำให้ GitHub
ขึ้นข้อความ *"Invalid Notebook"* พอดี ส่วนโน้ตบุ๊กสำหรับเทรนโมเดลมีขนาด 1.68 MB
ซึ่งเกือบทั้งหมดคือภาพผลลัพธ์ที่ฝังมาในรูปแบบ base64 และเกินเพดานการแสดงผลของ GitHub ที่ราว 1 MB

ตอนนี้โน้ตบุ๊กทุกไฟล์เป็น `nbformat 4.5` มี cell ID ครบ และล้าง output ที่ฝังอยู่ออกหมดแล้ว

## บทสรุป

ระบบนี้คือโมเดลตรวจจับ YOLO26 ที่เทรนขึ้นมาเฉพาะทางเพื่อหาไฟในภาพถ่ายทางอากาศ ต่อเข้ากับตัวติดตามวัตถุ
ที่ตามไฟแต่ละจุดไปตลอดเฟรมของวิดีโอ โดยมีวงจรการปฏิบัติงานล้อมรอบอยู่ซึ่งเป็นตัวทำให้ระบบมีประโยชน์จริง
ทั้งอากาศยานไร้คนขับที่กวาดพื้นที่ซึ่งไม่มีใครไปลาดตระเวนไหว
และศูนย์ควบคุมที่แปลงผลตรวจจับหนึ่งครั้งให้กลายเป็นการส่งชุดปฏิบัติการเข้าพื้นที่

องค์ประกอบแต่ละชิ้นไม่ได้พิสดารอะไรเลย สิ่งที่ทำให้แนวทางนี้ได้ผลคือ งานส่วนที่แพงที่สุด
อันได้แก่การเพ่งมองอย่างต่อเนื่องบนภูมิประเทศกว้างใหญ่และห่างไกล เป็นงานที่โมเดลทำได้ดีพอดี ส่วนคนทำได้ไม่ดี

### ไปต่อที่ไหนดี

เปิด **[`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb)** แล้วรันจากบนลงล่าง

1. **[ส่วนที่ 1](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-1)** เทรนโมเดลแล้วส่งออก `best.pt`
2. **[ส่วนที่ 2](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-2)** ทดสอบกับภาพนิ่ง
3. **[ส่วนที่ 3](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb#part-3)** ทดสอบกับวิดีโอพร้อมการติดตามวัตถุ

เรื่องที่น่าไปต่อหลังจากนั้น ได้แก่ การนำโมเดลไปติดตั้งใช้งานบนสถานีภาคพื้นดินด้วย
[`roboflow/inference`](https://github.com/roboflow/inference) แทนการเรียกผ่าน cloud API
การแยกชั้นข้อมูล `fire` ออกเป็นคลาสเปลวไฟกับคลาสควัน (ควันมองเห็นได้จากไกลกว่ามาก)
และการเอาหมายเลข track ไปใช้วัดอัตราการลุกลาม แทนที่จะรายงานแค่ว่าเจอไฟหรือไม่เจอ

### แหล่งอ้างอิงและเครดิต

- บทความต้นฉบับ: [Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/) โดย Timothy M., Roboflow Blog
- คลังโค้ดต้นทาง: [tim3in/Fire-Detection-Drone](https://github.com/tim3in/Fire-Detection-Drone)
- ชุดข้อมูล: [`drone-fire-detection-byija`](https://universe.roboflow.com/tim-4ijf0/drone-fire-detection-byija) บน Roboflow Universe

**รายการอ้างอิงชุดข้อมูล**

> Alireza Shamsoshoara, Fatemeh Afghah, Abolfazl Razi, Liming Zheng, Peter Fulé, Erik
> Blasch, November 19, 2020, "The FLAME dataset: Aerial Imagery Pile burn detection
> using drones (UAVs)", IEEE Dataport, doi: https://dx.doi.org/10.21227/qad6-r683

**รายการอ้างอิงบทความ**

> Timothy M. (Sep 19, 2023). Aerial Fire Detection with Drone Imagery and Computer
> Vision. Roboflow Blog: https://blog.roboflow.com/aerial-fire-detection/